In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, group
from pyspark.sql import functions as F

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat, src = cfg["catalog"], group.source(cfg, env)
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
f = lambda t: spark.table(f"{src}.gold.{t}")
print(cat, "group daily + intercompany, FQF from", src)

In [0]:
d, ic = g("fact_group_daily"), g("group_intercompany")
grp = group.load_group()
pt = grp["intercompany"]["bhg_payer_type"]

n = d.count()
check("one row per date, company and measure", d.select("date_key", "company_code", "measure").distinct().count() == n, n)
check("every date_key is in dim_date", d.join(g("dim_date"), "date_key", "left_anti").count() == 0)
known = {"bhg": {"billed", "tax", "collected"}, "fqf": {"disbursed", "interest_due", "collected", "fees"}}
seen = {(r[0], r[1]) for r in d.select("company_code", "measure").distinct().collect()}
check("only the designed measures", all(m in known.get(c, set()) for c, m in seen), sorted(seen))
check("group amount = amount - intercompany", d.filter("group_amount <> amount - intercompany").count() == 0)
ic_where = {(r[0], r[1]) for r in d.filter("intercompany <> 0").select("company_code", "measure").distinct().collect()}
check("intercompany only on BHG collected and FQF disbursed", ic_where <= {("bhg", "collected"), ("fqf", "disbursed")}, ic_where)

tot = lambda df: df.agg(F.sum("amount")).first()[0] or 0
bhg_total = tot(g("fact_bhg_event"))
check("BHG total = BHG Gold fact", abs(tot(d.filter("company_code = 'bhg'")) - bhg_total) < 0.01, bhg_total)
fe = f("fact_fqf_event")
fqf_total = fe.agg(F.sum(F.when(F.col("event_type") == "emi_due", F.coalesce("interest_part", F.lit(0))).otherwise(F.col("amount")))).first()[0]
check("FQF total = FQF Gold fact (instalments as interest)", abs(tot(d.filter("company_code = 'fqf'")) - fqf_total) < 0.01, fqf_total)

bhg_ic = d.filter("company_code = 'bhg'").agg(F.sum("intercompany")).first()[0] or 0
fqf_ic = d.filter("company_code = 'fqf'").agg(F.sum("intercompany")).first()[0] or 0
check("BHG intercompany = BHG payments by FQF loans", abs(bhg_ic - (ic.agg(F.sum("bhg_received")).first()[0] or 0)) < 0.01, bhg_ic)
check("FQF intercompany = FQF hospital-bill payouts", abs(fqf_ic - (ic.agg(F.sum("fqf_paid_out")).first()[0] or 0)) < 0.01, fqf_ic)

check("one intercompany row per invoice", ic.select("invoice_id").distinct().count() == ic.count())
check("matched invoices have no difference", ic.filter("status = 'matched' AND difference <> 0").count() == 0)
ic.groupBy("status").agg(F.count("*").alias("invoices"), F.sum("bhg_received").alias("bhg_received"),
                         F.sum("fqf_paid_out").alias("fqf_paid_out")).orderBy("status").show()
breaks = ic.filter("status <> 'matched'").count()
if env == "prod":
    check("prod: every intercompany invoice matches", breaks == 0, breaks)
else:
    print(f"INFO {breaks} intercompany breaks; expected below prod, because dev and test sample each company separately")

d.groupBy("company_code", "measure").agg(F.sum("amount").alias("amount"), F.sum("intercompany").alias("intercompany"),
                                          F.sum("group_amount").alias("group_amount")).orderBy("company_code", "measure").show()

print(f"\nt23_group_daily: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t23_group_daily failed")